### Config

In [0]:
# Retrieve secrets from Databricks Secret Scope
# Secrets are never displayed in output — shown as [REDACTED]
client_id     = dbutils.secrets.get(scope="africa-pulse-scope", key="client-id")
client_secret = dbutils.secrets.get(scope="africa-pulse-scope", key="client-secret")
tenant_id     = dbutils.secrets.get(scope="africa-pulse-scope", key="tenant-id")
storage_acct  = dbutils.secrets.get(scope="africa-pulse-scope", key="storage-account")

# Configure OAuth
spark.conf.set(f"fs.azure.account.auth.type.{storage_acct}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_acct}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_acct}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_acct}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_acct}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print("Authentication configured successfully")
print(f"Client ID: {client_id[:8]}...") # Shows only first 8 chars to confirm it loaded

### Set Up Path

In [0]:
# Cell 2 - Paths
BASE_PATH = "abfss://africa-economic-pulse@africapulse.dfs.core.windows.net"

# Silver paths (reading from)
SILVER_GDP = f"{BASE_PATH}/silver/gdp_cleaned/"
SILVER_INFLATION = f"{BASE_PATH}/silver/inflation_cleaned/"
SILVER_UNEMPLOYMENT = f"{BASE_PATH}/silver/unemployment_cleaned/"

# Gold paths (writing to)
GOLD_SUMMARY = f"{BASE_PATH}/gold/economic_summary/"
GOLD_NIGERIA = f"{BASE_PATH}/gold/nigeria_spotlight/"
GOLD_COMPARISON = f"{BASE_PATH}/gold/country_comparisons/"

print("Paths configured")

### Read Silver tables

In [0]:
# Cell 3 - Read Silver tables
from pyspark.sql.functions import col, avg, round, rank, lag
from pyspark.sql.window import Window

gdp_df = spark.read.format("delta").load(SILVER_GDP)
inflation_df = spark.read.format("delta").load(SILVER_INFLATION)
unemployment_df = spark.read.format("delta").load(SILVER_UNEMPLOYMENT)

print(f"GDP records: {gdp_df.count()}")
print(f"Inflation records: {inflation_df.count()}")
print(f"Unemployment records: {unemployment_df.count()}")

### Economic Summary

In [0]:
# Cell 4 - Create unified economic summary table
from pyspark.sql.functions import col, round, avg, when, lit

# Rename value columns before joining
gdp = gdp_df.select(
    "country_iso3", "country_name", "year",
    col("indicator_value").alias("gdp_usd")
)

inflation = inflation_df.select(
    "country_iso3", "year",
    col("indicator_value").alias("inflation_pct")
)

unemployment = unemployment_df.select(
    "country_iso3", "year",
    col("indicator_value").alias("unemployment_pct")
)

# Join all three on country and year
economic_summary = gdp\
    .join(inflation, ["country_iso3", "year"], "left")\
    .join(unemployment, ["country_iso3", "year"], "left")\
    .withColumn("gdp_billions", round(col("gdp_usd") / 1e9, 2))\
    .withColumn("data_quality_flag",
        when(col("gdp_usd").isNull(), "missing_gdp")
        .when(col("inflation_pct").isNull(), "missing_inflation")
        .when(col("unemployment_pct").isNull(), "missing_unemployment")
        .otherwise("complete")
    )

print(f"Economic summary records: {economic_summary.count()}")
display(economic_summary.filter(col("country_iso3") == "NGA").orderBy("year", ascending=False).limit(5))

### Nigeria Spotlight

In [0]:
# Cell 5 - Nigeria spotlight table
from pyspark.sql.functions import lag, round
from pyspark.sql.window import Window

window = Window.partitionBy("country_iso3").orderBy("year")

nigeria_spotlight = economic_summary\
    .filter(col("country_iso3") == "NGA")\
    .withColumn("gdp_prev_year", lag("gdp_billions", 1).over(window))\
    .withColumn("gdp_growth_pct", 
        round((col("gdp_billions") - col("gdp_prev_year")) / col("gdp_prev_year") * 100, 2)
    )\
    .orderBy("year", ascending=False)

display(nigeria_spotlight)

### Country Comparisons

In [0]:
# Cell 6 - Top 10 African economies by average GDP (2015-2023)
from pyspark.sql.functions import avg, round, desc

top_economies = economic_summary\
    .filter((col("year") >= 2015) & (col("year") <= 2023))\
    .filter(col("gdp_usd").isNotNull())\
    .groupBy("country_iso3", "country_name")\
    .agg(
        round(avg("gdp_billions"), 2).alias("avg_gdp_billions"),
        round(avg("inflation_pct"), 2).alias("avg_inflation_pct"),
        round(avg("unemployment_pct"), 2).alias("avg_unemployment_pct")
    )\
    .orderBy(desc("avg_gdp_billions"))

display(top_economies.limit(10))

### Write Gold as Parquet

In [0]:
from pyspark.sql.functions import current_date, lit, col, round, avg, lag, when
from pyspark.sql.window import Window


economic_summary_final = economic_summary\
    .withColumn("ingestion_date", current_date())\
    .withColumn("pipeline_name", lit("worldbank_gold_aggregation"))\
    .withColumn("source_system", lit("World Bank API v2"))

nigeria_spotlight_final = nigeria_spotlight\
    .withColumn("ingestion_date", current_date())\
    .withColumn("pipeline_name", lit("worldbank_gold_aggregation"))\
    .withColumn("source_system", lit("World Bank API v2"))

top_economies_final = top_economies\
    .withColumn("ingestion_date", current_date())\
    .withColumn("pipeline_name", lit("worldbank_gold_aggregation"))\
    .withColumn("source_system", lit("World Bank API v2"))

# Write WITHOUT partitionBy - keeps year as regular column
economic_summary_final.write\
    .format("parquet")\
    .mode("overwrite")\
    .save(GOLD_SUMMARY)

nigeria_spotlight_final.write\
    .format("parquet")\
    .mode("overwrite")\
    .save(GOLD_NIGERIA)

top_economies_final.write\
    .format("parquet")\
    .mode("overwrite")\
    .save(GOLD_COMPARISON)

print("✅ All Gold tables rewritten without partitioning")
print(f"Economic summary: {spark.read.parquet(GOLD_SUMMARY).count()} records")
print(f"Nigeria spotlight: {spark.read.parquet(GOLD_NIGERIA).count()} records")
print(f"Country comparisons: {spark.read.parquet(GOLD_COMPARISON).count()} records")